In [ ]:
"""
Enhanced Polynomial Root Classification via Machine Learning
Following methodology from Lample & Charton (2020) with additional statistical rigor

This framework evaluates ML models' ability to discover algebraic classification rules
for polynomial roots with proper statistical validation and class balance handling.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, 
                           roc_auc_score, roc_curve, auc, balanced_accuracy_score,
                           precision_recall_fscore_support, f1_score)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from imblearn.over_sampling import SMOTE
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Any, Optional
import time
import warnings
import psutil
import tracemalloc
from itertools import combinations
from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')

# Optional imports
AVAILABLE_MODELS = {}


class PolynomialRootClassifier:
    """
    Experimental framework for polynomial root classification.
    """
    
    def __init__(self, degree: int, n_samples: int = 10000, test_size: float = 0.2, 
                 random_state: int = 42, handle_imbalance: bool = True,
                 min_samples_per_class: int = 500):
        """
        Initialize polynomial root classification experiment.
        
        Args:
            degree: Polynomial degree (2, 3, 4, or 5)
            n_samples: Total number of samples to generate
            test_size: Proportion of data for testing
            random_state: Random seed for reproducibility
            handle_imbalance: Whether to handle class imbalance with SMOTE
            min_samples_per_class: Minimum samples per class for balanced generation
        """
        self.degree = degree
        self.n_samples = n_samples
        self.test_size = test_size
        self.random_state = random_state
        self.handle_imbalance = handle_imbalance
        self.min_samples_per_class = min_samples_per_class
        
        np.random.seed(random_state)
        
        # Data storage
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.coefficients = None
        self.labels = None
        
    def generate_data(self, coefficient_range: Tuple[float, float] = (-10, 10)) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate polynomial coefficients with balanced classes.
        
        Args:
            coefficient_range: Range for coefficient values
            
        Returns:
            Tuple of (coefficients, labels)
        """
        
        if self.degree == 2:
            return self._generate_quadratic_data(coefficient_range)
        elif self.degree == 3:
            return self._generate_cubic_data(coefficient_range)
        elif self.degree == 4:
            return self._generate_quartic_data(coefficient_range)
        elif self.degree == 5:
            return self._generate_quintic_data(coefficient_range)
        else:
            raise ValueError(f"Degree {self.degree} not supported")
    
    def _generate_balanced_samples(self, generator_func, n_classes: int, 
                                  coefficient_range: Tuple[float, float]) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate balanced samples for multi-class problems.
        
        Args:
            generator_func: Function to generate samples and classify
            n_classes: Number of classes
            coefficient_range: Range for coefficients
            
        Returns:
            Balanced coefficients and labels
        """
        samples_per_class = max(self.n_samples // n_classes, self.min_samples_per_class)
        class_samples = {i: [] for i in range(n_classes)}
        class_counts = {i: 0 for i in range(n_classes)}
        
        # Generate samples until we have enough for each class
        max_attempts = self.n_samples * 10
        attempts = 0
        
        while min(class_counts.values()) < samples_per_class and attempts < max_attempts:
            # Generate batch of samples
            coeffs, label = generator_func(coefficient_range)
            
            if class_counts[label] < samples_per_class:
                class_samples[label].append(coeffs)
                class_counts[label] += 1
            
            attempts += 1
        
        # Combine all samples
        all_coeffs = []
        all_labels = []
        
        for class_idx in range(n_classes):
            coeffs_list = class_samples[class_idx][:samples_per_class]
            all_coeffs.extend(coeffs_list)
            all_labels.extend([class_idx] * len(coeffs_list))
        
        return np.array(all_coeffs), np.array(all_labels)
    
    def _generate_quadratic_data(self, coefficient_range: Tuple[float, float] = (-10, 10)) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate quadratic polynomial data with balanced classes.
        """
        if self.handle_imbalance:
            # Generate balanced samples
            def generate_single(coef_range): 
                a = np.random.uniform(*coef_range)
                b = np.random.uniform(*coef_range)
                c = np.random.uniform(*coef_range)
                discriminant = b**2 - 4*a*c
                label = 1 if discriminant < 0 else 0
                return [a, b, c], label
            
            coefficients, labels = self._generate_balanced_samples(generate_single, 2, coefficient_range)
        else:
            # Original unbalanced generation
            a = np.random.uniform(*coefficient_range, self.n_samples)
            b = np.random.uniform(*coefficient_range, self.n_samples)
            c = np.random.uniform(*coefficient_range, self.n_samples)
            coefficients = np.column_stack([a, b, c])
            discriminant = b**2 - 4*a*c
            labels = (discriminant < 0).astype(int)
        
        self.coefficients = coefficients
        self.labels = labels
        self.feature_names = ['a', 'b', 'c']
        
        # Calculate invariants for analysis
        a = coefficients[:, 0]
        b = coefficients[:, 1]
        c = coefficients[:, 2]
        self.discriminant = b**2 - 4*a*c

        # Add b²/ac invariant
        ac = a * c
        epsilon = 1e-8
        self.b2_ac = np.where(np.abs(ac) > epsilon, b**2 / ac, 0)

        return coefficients, labels
    
    def _generate_cubic_data(self, coefficient_range: Tuple[float, float] = (-10, 10)) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate cubic polynomial data with balanced classes.
        """
        if self.handle_imbalance:
            def generate_single(coef_range):  # Add parameter here
                A = np.random.uniform(*coef_range)
                B = np.random.uniform(*coef_range)
                C = np.random.uniform(*coef_range)
                roots = np.roots([1, A, B, C])
                has_complex = np.any(np.abs(roots.imag) > 1e-10)
                label = 1 if has_complex else 0
                return [A, B, C], label
            
            coefficients, labels = self._generate_balanced_samples(generate_single, 2, coefficient_range)
        else:
            A = np.random.uniform(*coefficient_range, self.n_samples)
            B = np.random.uniform(*coefficient_range, self.n_samples)
            C = np.random.uniform(*coefficient_range, self.n_samples)
            coefficients = np.column_stack([A, B, C])
            
            labels = np.zeros(self.n_samples, dtype=int)
            for i in range(self.n_samples):
                roots = np.roots([1, A[i], B[i], C[i]])
                has_complex = np.any(np.abs(roots.imag) > 1e-10)
                labels[i] = 1 if has_complex else 0
        
        # Calculate cubic discriminant
        A = coefficients[:, 0]
        B = coefficients[:, 1] 
        C = coefficients[:, 2]
        self.cubic_discriminant = 18*A*B*C - 4*A**3*C + A**2*B**2 - 4*B**3 - 27*C**2
        
        self.coefficients = coefficients
        self.labels = labels
        self.feature_names = ['A', 'B', 'C']
        
        return coefficients, labels
    
    def _generate_quartic_data(self, coefficient_range: Tuple[float, float] = (-10, 10)) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate monic quartic polynomial data: x⁴ + Ax³ + Bx² + Cx + D
        Multi-class classification: 0 (4 real), 1 (2 real), 2 (0 real)
        """
        if self.handle_imbalance:
            def generate_single(coef_range):
                A = np.random.uniform(*coef_range)
                B = np.random.uniform(*coef_range)
                C = np.random.uniform(*coef_range)
                D = np.random.uniform(*coef_range)
                roots = np.roots([1, A, B, C, D])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                if n_real >= 3:
                    label = 0
                elif n_real >= 1:
                    label = 1
                else:
                    label = 2
                return [A, B, C, D], label
            
            coefficients, labels = self._generate_balanced_samples(generate_single, 3, coefficient_range)
        else:
            A = np.random.uniform(*coefficient_range, self.n_samples)
            B = np.random.uniform(*coefficient_range, self.n_samples)
            C = np.random.uniform(*coefficient_range, self.n_samples)
            D = np.random.uniform(*coefficient_range, self.n_samples)
            coefficients = np.column_stack([A, B, C, D])
            
            labels = np.zeros(self.n_samples, dtype=int)
            for i in range(self.n_samples):
                roots = np.roots([1, A[i], B[i], C[i], D[i]])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                if n_real >= 3:
                    labels[i] = 0
                elif n_real >= 1:
                    labels[i] = 1
                else:
                    labels[i] = 2
        
        # Calculate quartic invariants
        n_samples = len(coefficients)
        self.quartic_inv_I = np.zeros(n_samples)
        self.quartic_inv_J = np.zeros(n_samples)
        self.quartic_inv_Delta_expr = np.zeros(n_samples)
        self.quartic_inv_P = np.zeros(n_samples)
        self.quartic_inv_D_aux = np.zeros(n_samples)
        
        for i in range(n_samples):
            a, b, c, d, e = 1, coefficients[i, 0], coefficients[i, 1], coefficients[i, 2], coefficients[i, 3]
        
            # Classical quartic invariants
            I = 12*a*e - 3*b*d + c*c
            J = 72*a*c*e + 9*b*c*d - 27*a*d*d - 27*b*b*e - 2*c**3
            Delta_expr = 4*I**3 - J**2
            P = 8*a*c - 3*b*b
            D_aux = 64*a**3*e - 16*a*a*c*c + 16*a*b*b*c - 16*a*a*b*d - 3*b**4
            
            self.quartic_inv_I[i] = I
            self.quartic_inv_J[i] = J
            self.quartic_inv_Delta_expr[i] = Delta_expr
            self.quartic_inv_P[i] = P
            self.quartic_inv_D_aux[i] = D_aux
        
        self.coefficients = coefficients
        self.labels = labels
        self.feature_names = ['A', 'B', 'C', 'D']
        
        return coefficients, labels
    def _generate_quintic_data(self, coefficient_range: Tuple[float, float] = (-10, 10)) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate quintic polynomial data with balanced classes.
        """
        if self.handle_imbalance:
            def generate_single(coef_range): 
                A = np.random.uniform(*coef_range)
                B = np.random.uniform(*coef_range)
                C = np.random.uniform(*coef_range)
                D = np.random.uniform(*coef_range)
                E = np.random.uniform(*coef_range)
                roots = np.roots([1, A, B, C, D, E])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                if n_real == 5:
                    label = 0
                elif n_real == 3:
                    label = 1
                else:
                    label = 2
                return [A, B, C, D, E], label
            
            coefficients, labels = self._generate_balanced_samples(generate_single, 3, coefficient_range)
        else:
            A = np.random.uniform(*coefficient_range, self.n_samples)
            B = np.random.uniform(*coefficient_range, self.n_samples)
            C = np.random.uniform(*coefficient_range, self.n_samples)
            D = np.random.uniform(*coefficient_range, self.n_samples)
            E = np.random.uniform(*coefficient_range, self.n_samples)
            coefficients = np.column_stack([A, B, C, D, E])
            
            labels = np.zeros(self.n_samples, dtype=int)
            for i in range(self.n_samples):
                roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                if n_real == 5:
                    labels[i] = 0
                elif n_real == 3:
                    labels[i] = 1
                else:
                    labels[i] = 2
        
        # Calculate proper quintic Tschirnhaus invariants
        self._calculate_quintic_invariants(coefficients)
        
        self.coefficients = coefficients
        self.labels = labels
        self.feature_names = ['A', 'B', 'C', 'D', 'E']
        
        return coefficients, labels
    
    def _calculate_quintic_invariants(self, coefficients):
        """Calculate proper Tschirnhaus invariants for quintic."""
        n = len(coefficients)
        self.quintic_I2 = np.zeros(n)
        self.quintic_I3 = np.zeros(n)
        self.quintic_I4 = np.zeros(n)
        self.quintic_I5 = np.zeros(n)
        
        for i in range(n):
            a0, a1, a2, a3, a4, a5 = 1, coefficients[i, 0], coefficients[i, 1], coefficients[i, 2], coefficients[i, 3], coefficients[i, 4]
            
            # Tschirnhaus invariants (simplified but proper forms)
            self.quintic_I2[i] = a2 - (2*a1**2)/5
            self.quintic_I3[i] = a3 - (3*a1*a2)/5 + (2*a1**3)/25
            self.quintic_I4[i] = a4 - (2*a1*a3)/5 + (a1**2*a2)/5 - (a1**4)/125
            self.quintic_I5[i] = a5 - (a1*a4)/5 + (a1**2*a3)/25 - (a1**3*a2)/125 + (a1**5)/3125
    
    def create_feature_sets(self) -> Dict[str, np.ndarray]:
        """
        Create feature sets with varying levels of domain knowledge.
        """
        if self.coefficients is None:
            raise ValueError("Must generate data first")
        
        feature_sets = {}
        
        # 1. Raw coefficients only
        feature_sets['raw_coefficients'] = self.coefficients
        
        # 2. Polynomial features degree 2
        poly = PolynomialFeatures(degree=2, include_bias=False)
        feature_sets['polynomial_deg2'] = poly.fit_transform(self.coefficients)
        
        # 3. Domain-informed features with invariants
        if self.degree == 2:
            # Calculate b2_ac invariant
            a = self.coefficients[:, 0]
            b = self.coefficients[:, 1]
            c = self.coefficients[:, 2]
            ac = a * c
            epsilon = 1e-8
            b2_ac = np.where(np.abs(ac) > epsilon, b**2 / ac, 0)
            
            # Add b2_ac invariant feature set
            feature_sets['with_invariants'] = np.column_stack([
                self.coefficients, b2_ac.reshape(-1, 1)
            ])
        elif self.degree == 3:
            # Cubic discriminant as invariant
            feature_sets['with_invariants'] = np.column_stack([
                self.coefficients, self.cubic_discriminant.reshape(-1, 1)
            ])
        elif self.degree == 4:
            # Use the original five quartic invariants
            feature_sets['with_invariants'] = np.column_stack([
                self.coefficients,
                self.quartic_inv_I.reshape(-1, 1),
                self.quartic_inv_J.reshape(-1, 1),
                self.quartic_inv_Delta_expr.reshape(-1, 1),
                self.quartic_inv_P.reshape(-1, 1),
                self.quartic_inv_D_aux.reshape(-1, 1)
            ])
        elif self.degree == 5:
            # Proper Tschirnhaus invariants for quintic
            feature_sets['with_invariants'] = np.column_stack([
                self.coefficients,
                self.quintic_I2.reshape(-1, 1),
                self.quintic_I3.reshape(-1, 1),
                self.quintic_I4.reshape(-1, 1),
                self.quintic_I5.reshape(-1, 1)
            ])
        
        return feature_sets
    
    def get_models(self) -> Dict[str, Any]:
        """
        Initialize ML models including baseline.
        """
        models = {}
        
        # Baseline model
        models['MajorityClass'] = DummyClassifier(strategy='most_frequent', random_state=self.random_state)
        
        # CART
        models['CART'] = DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=50,
            min_samples_leaf=20,
            random_state=self.random_state
        )
        
        # Logistic Regression
        models['LogisticRegression'] = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                max_iter=1000,
                random_state=self.random_state,
                solver='lbfgs'
            ))
        ])
        
        # SVM RBF
        models['SVM_RBF'] = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                random_state=self.random_state,
                probability=True  # Needed for ROC curves
            ))
        ])
        
        # Neural Network
        models['NeuralNetwork'] = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', MLPClassifier(
                hidden_layer_sizes=(100, 50),
                activation='relu',
                solver='adam',
                alpha=0.0001,
                learning_rate='constant',
                learning_rate_init=0.001,
                max_iter=500,
                random_state=self.random_state,
                early_stopping=True,
                validation_fraction=0.15
            ))
        ])
        
        # Random Forest
        models['RandomForest'] = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            max_features='sqrt',
            random_state=self.random_state,
            n_jobs=-1
        )
        
        # Gradient Boosting
        models['GradientBoosting'] = GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            subsample=0.8,
            random_state=self.random_state
        )
        
        # XGBoost
        if AVAILABLE_MODELS.get('xgboost'):
            if self.degree == 2 or self.degree == 3:
                models['XGBoost'] = XGBClassifier(
                    n_estimators=100,
                    max_depth=5,
                    learning_rate=0.1,
                    random_state=self.random_state,
                    use_label_encoder=False,
                    eval_metric='logloss'
                )
            else:
                models['XGBoost'] = XGBClassifier(
                    n_estimators=100,
                    max_depth=5,
                    learning_rate=0.1,
                    random_state=self.random_state,
                    use_label_encoder=False,
                    objective='multi:softmax',
                    eval_metric='mlogloss'
                )
        
        return models
    
    def evaluate_model_cv(self, model: Any, X: np.ndarray, y: np.ndarray, 
                         cv_folds: int = 5) -> Dict[str, Any]:
        """
        Evaluate model using cross-validation with comprehensive metrics.
        """
        # Start tracking memory
        tracemalloc.start()
        process = psutil.Process()
        start_memory = process.memory_info().rss / 1024 / 1024  # MB
        
        # Cross-validation
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=self.random_state)
        
        # Metrics storage
        cv_scores = []
        cv_balanced_scores = []
        cv_f1_macro = []
        cv_f1_micro = []
        train_times = []
        predict_times = []
        
        for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            # Training time
            start_time = time.time()
            model.fit(X_train, y_train)
            train_time = time.time() - start_time
            train_times.append(train_time)
            
            # Prediction time
            start_time = time.time()
            y_pred = model.predict(X_val)
            predict_time = time.time() - start_time
            predict_times.append(predict_time)
            
            # Metrics
            cv_scores.append(accuracy_score(y_val, y_pred))
            cv_balanced_scores.append(balanced_accuracy_score(y_val, y_pred))
            cv_f1_macro.append(f1_score(y_val, y_pred, average='macro'))
            cv_f1_micro.append(f1_score(y_val, y_pred, average='micro'))
        
        # Memory usage
        current_memory = process.memory_info().rss / 1024 / 1024
        memory_used = current_memory - start_memory
        tracemalloc.stop()
        
        # Final training on full data for other metrics
        model.fit(X, y)
        
        results = {
            'accuracy_mean': np.mean(cv_scores),
            'accuracy_std': np.std(cv_scores),
            'balanced_accuracy_mean': np.mean(cv_balanced_scores),
            'balanced_accuracy_std': np.std(cv_balanced_scores),
            'f1_macro_mean': np.mean(cv_f1_macro),
            'f1_macro_std': np.std(cv_f1_macro),
            'f1_micro_mean': np.mean(cv_f1_micro),
            'f1_micro_std': np.std(cv_f1_micro),
            'train_time_mean': np.mean(train_times),
            'train_time_std': np.std(train_times),
            'predict_time_mean': np.mean(predict_times),
            'predict_time_std': np.std(predict_times),
            'memory_mb': memory_used,
            'model': model
        }
        
        return results
    
    def analyze_invariant_correlation(self) -> pd.DataFrame:
        """
        Analyze correlation between invariants and classification labels.
        """
        correlations = []
        
        if self.degree == 2:
            corr = np.corrcoef(self.discriminant, self.labels)[0, 1]
            correlations.append({'Invariant': 'Discriminant', 'Correlation': corr})
            
        elif self.degree == 3:
            corr = np.corrcoef(self.cubic_discriminant, self.labels)[0, 1]
            correlations.append({'Invariant': 'Cubic Discriminant', 'Correlation': corr})
            
        elif self.degree == 4:
            # Back to original invariant names
            for inv_name, inv_values in [
                ('I', self.quartic_inv_I), 
                ('J', self.quartic_inv_J), 
                ('Delta_expr', self.quartic_inv_Delta_expr),
                ('P', self.quartic_inv_P),
                ('D_aux', self.quartic_inv_D_aux)
            ]:
                corr = np.corrcoef(inv_values, self.labels)[0, 1]
                correlations.append({'Invariant': f'Quartic {inv_name}', 'Correlation': corr})
            
        elif self.degree == 5:
            for inv_name, inv_values in [
                ('I2', self.quintic_I2), 
                ('I3', self.quintic_I3), 
                ('I4', self.quintic_I4), 
                ('I5', self.quintic_I5)
            ]:
                corr = np.corrcoef(inv_values, self.labels)[0, 1]
                correlations.append({'Invariant': f'Quintic {inv_name}', 'Correlation': corr})
        
        return pd.DataFrame(correlations)
    
    def visualize_pca_space(self) -> None:
        """
        Visualize coefficient space using PCA colored by class.
        """
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(self.coefficients)
        
        plt.figure(figsize=(10, 6))
        scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=self.labels, 
                            cmap='viridis', alpha=0.6, s=10)
        plt.colorbar(scatter)
        plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
        plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
        plt.title(f'PCA of Degree {self.degree} Polynomial Coefficients')
        plt.show()
        
        return pca.explained_variance_ratio_
    
    def plot_confusion_matrix(self, y_true, y_pred, title="Confusion Matrix"):
        """
        Plot confusion matrix.
        """
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(title)
        plt.show()
        return cm
    
    def plot_roc_curves(self, models_results: Dict, X_test, y_test):
        """
        Plot ROC curves for binary classification.
        """
        if self.degree not in [2, 3]:  
            return
        
        plt.figure(figsize=(10, 8))
        
        for model_name, results in models_results.items():
            if 'model' not in results:
                continue
                
            model = results['model']
            
            # Get probabilities
            if hasattr(model, 'predict_proba'):
                y_prob = model.predict_proba(X_test)[:, 1]
            elif hasattr(model, 'decision_function'):
                y_prob = model.decision_function(X_test)
            else:
                continue
            
            fpr, tpr, _ = roc_curve(y_test, y_prob)
            auc_score = auc(fpr, tpr)
            
            plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.3f})')
        
        plt.plot([0, 1], [0, 1], 'k--', label='Random')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves - Degree {self.degree} Polynomial')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    
    def analyze_with_shap(self, model, X_train, feature_names):
        """
        Use SHAP to understand what the neural network learned.
        """
        if not AVAILABLE_MODELS.get('shap'):
            print("SHAP not available for analysis")
            return
        
        if hasattr(model, 'named_steps'):
            clf = model.named_steps['clf']
            scaler = model.named_steps['scaler']
            X_scaled = scaler.transform(X_train[:100])  # Use subset for speed
        else:
            clf = model
            X_scaled = X_train[:100]
        
        try:
            explainer = shap.Explainer(clf.predict, X_scaled)
            shap_values = explainer(X_scaled)
            
            # Summary plot
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_values, X_scaled, feature_names=feature_names, show=False)
            plt.title(f'SHAP Feature Importance - Degree {self.degree}')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"SHAP analysis failed: {e}")
    
    def run_multi_seed_experiment(self, n_seeds: int = 3) -> pd.DataFrame:
        """
        Run experiment with multiple random seeds for statistical rigor.
        """
        print(f"\n{'='*70}")
        print(f"Polynomial root classification - degree {self.degree}")
        print(f"Running {n_seeds} seeds with cross-validation")
        print(f"{'='*70}")
        
        all_results = []
        
        for seed in range(42, 42 + n_seeds):
            print(f"\n--- Seed {seed} ---")
            np.random.seed(seed)
            
            # Generate data
            X, y = self.generate_data()
            print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
            
            # Create feature sets
            feature_sets = self.create_feature_sets()
            
            # Initialize models
            models = self.get_models()
            
            for feature_name, X_features in feature_sets.items():
                print(f"\nFeature set: {feature_name} ({X_features.shape[1]} features)")
                
                for model_name, model in models.items():
                    print(f"  {model_name:20s}... ", end='')
                    
                    try:
                        if model_name == 'MajorityClass':
                            model = DummyClassifier(strategy='most_frequent', random_state=seed)
                        elif model_name == 'CART':
                            model = DecisionTreeClassifier(max_depth=5, random_state=seed)
                        elif model_name == 'RandomForest':
                            model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed, n_jobs=-1)
                        elif model_name == 'GradientBoosting':
                            model = GradientBoostingClassifier(n_estimators=100, random_state=seed)
                        elif model_name == 'XGBoost' and AVAILABLE_MODELS.get('xgboost'):
                            if self.degree in [2, 3]:
                                model = XGBClassifier(n_estimators=100, random_state=seed, use_label_encoder=False, eval_metric='logloss')
                            else:
                                model = XGBClassifier(n_estimators=100, random_state=seed, use_label_encoder=False, objective='multi:softmax', eval_metric='mlogloss')
                        
                        results = self.evaluate_model_cv(model, X_features, y, cv_folds=5)
                        
                        result_entry = {
                            'seed': seed,
                            'feature_set': feature_name,
                            'model': model_name,
                            'accuracy_mean': results['accuracy_mean'],
                            'accuracy_std': results['accuracy_std'],
                            'balanced_accuracy_mean': results['balanced_accuracy_mean'],
                            'balanced_accuracy_std': results['balanced_accuracy_std'],
                            'f1_macro_mean': results['f1_macro_mean'],
                            'train_time_mean': results['train_time_mean'],
                            'predict_time_mean': results['predict_time_mean'],
                            'memory_mb': results['memory_mb']
                        }
                        
                        all_results.append(result_entry)
                        
                        print(f"Acc: {results['accuracy_mean']:.3f}±{results['accuracy_std']:.3f}, "
                             f"Bal: {results['balanced_accuracy_mean']:.3f}±{results['balanced_accuracy_std']:.3f}, "
                             f"F1: {results['f1_macro_mean']:.3f}")
                        
                    except Exception as e:
                        print(f"FAILED: {str(e)[:50]}")
        
        return pd.DataFrame(all_results)
    
    def statistical_comparison(self, results_df: pd.DataFrame) -> pd.DataFrame:
        """
        Perform statistical significance tests between methods.
        """
        # Group by feature set and model
        grouped = results_df.groupby(['feature_set', 'model'])['accuracy_mean'].agg(['mean', 'std', 'count'])
            
        # Find best model for each feature set
        best_models = {}
        for feature_set in results_df['feature_set'].unique():
            subset = results_df[results_df['feature_set'] == feature_set]
            best = subset.groupby('model')['accuracy_mean'].mean().idxmax()
            best_models[feature_set] = best
        
        # Statistical tests
        significance_tests = []
        
        for feature_set in results_df['feature_set'].unique():
            subset = results_df[results_df['feature_set'] == feature_set]
            best_model = best_models[feature_set]
            best_scores = subset[subset['model'] == best_model]['accuracy_mean'].values
            
            for model in subset['model'].unique():
                if model != best_model:
                    model_scores = subset[subset['model'] == model]['accuracy_mean'].values
                    
                    # Paired t-test if same number of seeds
                    if len(best_scores) == len(model_scores) and len(best_scores) > 1:
                        t_stat, p_value = stats.ttest_rel(best_scores, model_scores)
                    else:
                        t_stat, p_value = stats.ttest_ind(best_scores, model_scores)
                    
                    significance_tests.append({
                        'feature_set': feature_set,
                        'best_model': best_model,
                        'compared_model': model,
                        'p_value': p_value,
                        'significant': p_value < 0.05
                    })
        
        return pd.DataFrame(significance_tests)


def display_detailed_metrics(results_df: pd.DataFrame, degree: int):
    """
    Display detailed metrics including computation times with 95% Confidence Intervals.
    """
    print(f"\n{'='*80}")
    print(f" DETAILED METRICS - DEGREE {degree} (95% CI)")
    print(f"{'='*80}")
    
    # Group by feature set and model
    for feature_set in results_df['feature_set'].unique():
        print(f"\n--- {feature_set} ---")
        subset = results_df[results_df['feature_set'] == feature_set]
        
        summary = subset.groupby('model').agg({
            'accuracy_mean': ['mean', 'std', 'count'],
            'balanced_accuracy_mean': ['mean', 'std'],
            'f1_macro_mean': ['mean', 'std'],
            'train_time_mean': ['mean', 'std'],
            'predict_time_mean': ['mean', 'std'],
            'memory_mb': ['mean', 'std']
        })
        
        print("\nModel Performance & Timing:")
        print("-" * 60)
        for model in summary.index:
            
            n_seeds = int(summary.loc[model, ('accuracy_mean', 'count')])
            t_stat = stats.t.ppf(0.975, df=n_seeds-1) if n_seeds > 1 else 0.0
            
            # Standard Accuracy
            acc = summary.loc[model, ('accuracy_mean', 'mean')]
            acc_std = summary.loc[model, ('accuracy_mean', 'std')]
            acc_ci = t_stat * (acc_std / np.sqrt(n_seeds)) if n_seeds > 1 else 0.0
            
            # Balanced Accuracy
            bal = summary.loc[model, ('balanced_accuracy_mean', 'mean')]
            bal_std = summary.loc[model, ('balanced_accuracy_mean', 'std')]
            bal_ci = t_stat * (bal_std / np.sqrt(n_seeds)) if n_seeds > 1 else 0.0
            
            train_t = summary.loc[model, ('train_time_mean', 'mean')]
            train_t_std = summary.loc[model, ('train_time_mean', 'std')]
            train_ci = t_stat * (train_t_std / np.sqrt(n_seeds)) if n_seeds > 1 else 0.0
            
            pred_t = summary.loc[model, ('predict_time_mean', 'mean')] * 1000  # Convert to ms
            mem = summary.loc[model, ('memory_mb', 'mean')]
            
            print(f"{model:20s}: Acc={acc:.3f}±{acc_ci:.3f}, "
                  f"Bal={bal:.3f}±{bal_ci:.3f}, " 
                  f"Train={train_t:.2f}±{train_ci:.2f}s, "
                  f"Pred={pred_t:.2f}ms, Mem={mem:.1f}MB")


def run_complete_analysis():
    """
    Run comprehensive analysis with all enhancements.
    """
    print("\n" + "="*80)
    print(" Polynomial root classification analysis")
    print("="*80)
    
    all_results = {}
    start_time = time.time()
    
    # Main experiments with multiple seeds
    for degree in [2, 3, 4, 5]:
        print(f"\n\n{'#'*70}")
        print(f" Degree {degree} polynomial analysis")
        print(f"{'#'*70}")
        
        # Initialize classifier
        classifier = PolynomialRootClassifier(
            degree=degree,
            n_samples=10000 if degree < 5 else 15000,
            test_size=0.2,
            random_state=42,
            handle_imbalance=False,
            min_samples_per_class=500
        )
        
        # Generate initial data for visualizations
        X, y = classifier.generate_data()
        
        # Invariant correlation analysis
        print("\n--- Invariant Correlation Analysis ---")
        corr_df = classifier.analyze_invariant_correlation()
        print(corr_df)
        
        # PCA visualization
        print("\n--- PCA Visualization ---")
        explained_variance = classifier.visualize_pca_space()
        print(f"Explained variance: {explained_variance}")
        
        # Run multi-seed experiment
        results_df = classifier.run_multi_seed_experiment(n_seeds=3)
        all_results[f'degree_{degree}'] = results_df

        display_detailed_metrics(results_df, degree)
        
        # Statistical comparison
        print("\n--- Statistical Significance Tests ---")
        sig_tests = classifier.statistical_comparison(results_df)
        print(f"Significant differences found: {sig_tests['significant'].sum()}/{len(sig_tests)}")
        
        print("\n--- Best Models Summary (95% CI) ---")
        for feature_set in results_df['feature_set'].unique():
            print(f"\n{feature_set}:")
            subset = results_df[results_df['feature_set'] == feature_set]
            
            best_models = []
            for model in subset['model'].unique():
                m_sub = subset[subset['model'] == model]
                n_count = len(m_sub)
                if n_count == 0: continue
                
                t_stat = stats.t.ppf(0.975, df=n_count-1) if n_count > 1 else 0
                
                bal_mean = m_sub['balanced_accuracy_mean'].mean()
                bal_ci = t_stat * (m_sub['balanced_accuracy_mean'].std() / np.sqrt(n_count)) if n_count > 1 else 0.0
                
                acc_mean = m_sub['accuracy_mean'].mean()
                acc_ci = t_stat * (m_sub['accuracy_mean'].std() / np.sqrt(n_count)) if n_count > 1 else 0.0
                
                best_models.append({
                    'Model': model,
                    'Balanced Acc': f"{bal_mean:.3f}±{bal_ci:.3f}",
                    'Standard Acc': f"{acc_mean:.3f}±{acc_ci:.3f}",
                    '_sort_val': bal_mean 
                })
            
            best_models.sort(key=lambda x: x['_sort_val'], reverse=True)
            for bm in best_models[:3]:
                print(f"  {bm['Model']:20s} | Bal: {bm['Balanced Acc']} | Acc: {bm['Standard Acc']}")
        
        # SHAP analysis for best neural network
        if AVAILABLE_MODELS.get('shap') and degree == 2:  # Do SHAP for quadratic as example
            print("\n--- SHAP Analysis ---")
            feature_sets = classifier.create_feature_sets()
            X_raw = feature_sets['raw_coefficients']
            nn_model = classifier.get_models()['NeuralNetwork']
            nn_model.fit(X_raw, y)
            classifier.analyze_with_shap(nn_model, X_raw, classifier.feature_names)
        
        # Store classifier
        all_results[f'classifier_{degree}'] = classifier
    
    # Cross-degree comparison with proper metrics
    print("\n\n" + "="*80)
    print(" Comparison with balanced metrics")
    print("="*80)
    
    comparison_data = []
    
    for degree in [2, 3, 4, 5]:
        df = all_results[f'degree_{degree}']
        
        # Best without domain knowledge
        raw = df[df['feature_set'] == 'raw_coefficients']
        if not raw.empty:
            best_raw = raw.groupby('model')['balanced_accuracy_mean'].mean().sort_values(ascending=False)
            best_raw_model = best_raw.index[0]
            best_raw_score = best_raw.iloc[0]
            
            raw_data = raw[raw['model'] == best_raw_model]['balanced_accuracy_mean']
            n_raw = len(raw_data)
            t_raw = stats.t.ppf(0.975, df=n_raw-1) if n_raw > 1 else 0
            best_raw_ci = t_raw * (raw_data.std() / np.sqrt(n_raw)) if n_raw > 1 else 0.0
            
            enhanced = df[df['feature_set'].str.contains('invariants', na=False)]
            if not enhanced.empty:
                best_enh = enhanced.groupby('model')['balanced_accuracy_mean'].mean().sort_values(ascending=False)
                best_enh_model = best_enh.index[0]
                best_enh_score = best_enh.iloc[0]
                
                enh_data = enhanced[enhanced['model'] == best_enh_model]['balanced_accuracy_mean']
                n_enh = len(enh_data)
                t_enh = stats.t.ppf(0.975, df=n_enh-1) if n_enh > 1 else 0
                best_enh_ci = t_enh * (enh_data.std() / np.sqrt(n_enh)) if n_enh > 1 else 0.0
            else:
                best_enh_model = best_raw_model
                best_enh_score = best_raw_score
                best_enh_ci = best_raw_ci
            
            comparison_data.append({
                'Degree': degree,
                'Best_Raw_Model': best_raw_model,
                'Raw_Balanced_Acc': f"{best_raw_score:.3f}±{best_raw_ci:.3f}", 
                'Best_Enhanced_Model': best_enh_model,
                'Enhanced_Balanced_Acc': f"{best_enh_score:.3f}±{best_enh_ci:.3f}", 
                'Improvement': f"{(best_enh_score - best_raw_score):.3f}"
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\nFinal Comparison Table:")
    print(comparison_df.to_string(index=False))
    
    # Total runtime
    total_time = time.time() - start_time
    print(f"\n\nTotal experiment runtime: {total_time/60:.2f} minutes")
    
    return all_results


# Main execution
if __name__ == "__main__":
    plt.style.use('seaborn-v0_8-whitegrid')
    sns.set_palette("husl")
    
    # Run complete analysis
    results = run_complete_analysis()
    

    print(" Analysis complete")